# Ergebnisse – Experiment-Summary
- Pro Experiment & Datensatz eine Tabelle (mit `n_runs` bei Exp 1/2/4); Bar-Chart bei Exp 1/2/4, bei Exp 3 SHAP-Tabelle + Bar-Charts (Top-Features und Text vs. numerisch)
- Quelle: alle MLflow-Runs aus `./mlruns`; fehlende/leere Experimente werden übersprungen

In [1]:
import os
import mlflow
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RESULT_DIR = "result_tables"
DIAGRAM_DIR = "diagrams"
os.makedirs(RESULT_DIR, exist_ok=True)
os.makedirs(DIAGRAM_DIR, exist_ok=True)
DS_ABBR = {"fake_jobs": "fj", "airbnb_paris": "airbnb"}

## Runs laden
- Nur vorhandene Experimente aus `./mlruns`; leere/fehlende werden übersprungen

In [2]:
mlflow.set_tracking_uri("file:./mlruns")

experiments = [f"{ds}_experiment_{i}" for ds in ["fake_jobs", "airbnb_paris"] for i in [1, 2, 3, 4]]

parts = []
for name in experiments:
    exp = mlflow.get_experiment_by_name(name)
    if exp is None:
        continue
    runs = mlflow.search_runs([exp.experiment_id])
    if len(runs) == 0:
        continue
    runs["experiment"] = name
    parts.append(runs)

df = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
print(f"{len(df)} Runs aus {len(parts)} Experimenten geladen")

/home/debian/TFM_master_thesis/.venv/lib/python3.14/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


440 Runs aus 8 Experimenten geladen


## Aufbereiten
- Datensatz / Experiment-Nr. / Modellname ableiten; Metrik-Spalten vereinheitlichen

In [3]:
datasets = ["fake_jobs", "airbnb_paris"]

if len(df):
    df["exp_num"] = df["experiment"].str.split("_").str[-1].astype(int)
    df["dataset"] = df["experiment"].str.rsplit("_experiment_", n=1).str[0]
    df["model"] = df["tags.mlflow.runName"] if "tags.mlflow.runName" in df.columns else df["run_id"]
    df = df[df["model"] != "contexttab_fixed"]  # veralteten Run ausschließen
    df = df.rename(columns={"metrics.auprc": "AUPRC", "metrics.auc_roc": "AUROC", "metrics.runtime_s": "runtime_s"})
    for col in ["AUPRC", "AUROC", "runtime_s"]:
        if col not in df.columns:
            df[col] = np.nan
    print("Datensätze:", df["dataset"].unique().tolist(), "| Experimente:", sorted(df["exp_num"].unique()))

Datensätze: ['fake_jobs', 'airbnb_paris'] | Experimente: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]


## Experiment 1 – Unsupervised Outlier Detection
- TFMs (TabPFN unsup., AnoLLM, FoMo-OD) vs. PyOD-Baselines; AUPRC & AUROC je Modell
- Tabelle je Datensatz; je 1 kombinierter Chart für AUPRC, AUROC und Runtime (beide Datensätze, Modelle alphabetisch)

In [ ]:
sub = df[df["exp_num"] == 1] if len(df) else df
if not len(sub):
    print("keine Exp-1-Runs")
else:
    # Tabelle je Datensatz
    for ds in datasets:
        s = sub[sub["dataset"] == ds]
        if not len(s):
            continue
        g = s.groupby("model")
        tab = g[["AUPRC", "AUROC", "runtime_s"]].mean()
        tab["n_runs"] = g.size()
        tab = tab.sort_values("AUPRC", ascending=False).round(4)
        save = tab.drop(columns="n_runs")  # n_runs nur zur Anzeige, nicht speichern
        save.to_csv(f"{RESULT_DIR}/exp1_{ds}.csv")
        # scientific-style .tex: booktabs (hrules) + fette Spaltenheader
        sty = (save.style.format(precision=4, escape="latex")
               .format_index(escape="latex", axis="index").format_index(escape="latex", axis="columns")
               .map_index(lambda v: "font-weight: bold;", axis="columns")
               .hide(axis="index", names=True).hide(axis="columns", names=True))
        sty.to_latex(f"{RESULT_DIR}/exp1_{ds}.tex", hrules=True, convert_css=True, position_float="centering",
                     caption=f"Experiment 1 -- Unsupervised Outlier Detection ({ds.replace('_', ' ')}): AUPRC, AUROC und Laufzeit je Modell.", label=f"tab:exp1_{ds}")
        print(f"=== Experiment 1 – {ds} ===")
        display(tab)

    # je 1 Chart (beide Datensätze, Modelle alphabetisch) -> diagrams/
    for metric, label, fname in [("AUPRC", "AUPRC", "auprc"), ("AUROC", "AUROC", "auroc"), ("runtime_s", "Runtime (s)", "runtime")]:
        piv = sub.pivot_table(index="model", columns="dataset", values=metric, aggfunc="mean").sort_index()
        ax = piv.plot.bar(figsize=(10, 4), rot=30)
        ax.set_title(f"Experiment 1 – {label} (beide Datensätze)")
        ax.set_ylabel(label)
        ax.set_xlabel("")
        ax.legend(title="Datensatz")
        plt.tight_layout()
        plt.savefig(f"{DIAGRAM_DIR}/exp1_{fname}.png", facecolor='white', edgecolor='white', dpi=300)
        plt.savefig(f"{DIAGRAM_DIR}/exp1_{fname}.pdf", facecolor='white', edgecolor='white')
        plt.show()
        plt.close()

## Experiment 2 – Enhanced Baseline-Modelle
- Baselines auf cleaned vs. semantisch vs. fastText vs. enhanced vs. enhanced+semantisch (AUPRC & AUROC je Detektor)
- Je Datensatz je 1 Chart für AUPRC und AUROC
- Hinweis: enhanced ist semi-supervised (Label-Leakage), nicht direkt mit unsupervised vergleichbar

In [ ]:
# feste Repräsentations-Reihenfolge (fastText neben den semantic-Analoga)
REP_ORDER = ["cleaned", "semantic_pca30", "semantic_pca100", "fast_text_pca30", "fast_text_pca100", "enhanced", "enhanced_pca30", "enhanced_semantic_pca30"]

for ds in datasets:
    sub = df[(df["exp_num"] == 2) & (df["dataset"] == ds)] if len(df) else df
    if not len(sub):
        print(f"{ds}: keine Exp-2-Runs")
        continue
    # Tabelle: AUPRC-Mittel + n_runs je Detektor × Repräsentation
    piv = sub.pivot_table(index="params.detector", columns="params.representation", values="AUPRC", aggfunc=["mean", "size"]).round(4)
    piv = piv.rename(columns={"mean": "AUPRC", "size": "n_runs"}, level=0)
    order = [r for r in REP_ORDER if r in piv.columns.get_level_values(1)]
    order += [r for r in piv.columns.get_level_values(1).unique() if r not in order]  # unbekannte Reps ans Ende
    piv = piv.reindex(order, axis=1, level=1)
    save = piv["AUPRC"]  # nur AUPRC speichern (ohne n_runs)
    save.to_csv(f"{RESULT_DIR}/exp2_{ds}.csv")
    # scientific-style .tex: booktabs (hrules) + fette Spaltenheader
    sty = (save.style.format(precision=4, escape="latex", na_rep="--")
           .format_index(escape="latex", axis="index").format_index(escape="latex", axis="columns")
           .map_index(lambda v: "font-weight: bold;", axis="columns")
           .hide(axis="index", names=True).hide(axis="columns", names=True))
    sty.to_latex(f"{RESULT_DIR}/exp2_{ds}.tex", hrules=True, convert_css=True, position_float="centering",
                 caption=f"Experiment 2 -- Enhanced Baselines ({ds.replace('_', ' ')}): AUPRC je Detektor und Repräsentation.", label=f"tab:exp2_{ds}")
    print(f"=== Experiment 2 – {ds} (AUPRC & n_runs je Detektor × Repräsentation) ===")
    display(piv)

    # AUPRC je Detektor × Repräsentation -> diagrams/
    ax = piv["AUPRC"].plot.bar(figsize=(10, 4), rot=0)
    ax.set_title(f"Experiment 2 – {ds}: AUPRC je Repräsentation")
    ax.set_ylabel("AUPRC")
    ax.set_xlabel("Detektor")
    ax.legend(title="Repräsentation", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.savefig(f"{DIAGRAM_DIR}/exp2_auprc_{DS_ABBR[ds]}.png", facecolor='white', edgecolor='white', dpi=300)
    plt.savefig(f"{DIAGRAM_DIR}/exp2_auprc_{DS_ABBR[ds]}.pdf", facecolor='white', edgecolor='white')
    plt.show()
    plt.close()

    # AUROC je Detektor × Repräsentation -> diagrams/
    piv_auc = sub.pivot_table(index="params.detector", columns="params.representation", values="AUROC", aggfunc="mean").round(4)
    piv_auc = piv_auc.reindex(columns=[r for r in order if r in piv_auc.columns])
    ax = piv_auc.plot.bar(figsize=(10, 4), rot=0)
    ax.set_title(f"Experiment 2 – {ds}: AUROC je Repräsentation")
    ax.set_ylabel("AUROC")
    ax.set_xlabel("Detektor")
    ax.legend(title="Repräsentation", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.savefig(f"{DIAGRAM_DIR}/exp2_auroc_{DS_ABBR[ds]}.png", facecolor='white', edgecolor='white', dpi=300)
    plt.savefig(f"{DIAGRAM_DIR}/exp2_auroc_{DS_ABBR[ds]}.pdf", facecolor='white', edgecolor='white')
    plt.show()
    plt.close()

## Experiment 3 – Semantische Relevanz (SHAP)
- ConTextTab-SHAP: wichtigste Features je Datensatz (Tabelle + Bar-Charts: Top-Features und Text vs. numerisch)

In [ ]:
for ds in datasets:
    sub = df[(df["exp_num"] == 3) & (df["dataset"] == ds)] if len(df) else df
    if not len(sub):
        continue
    rel_cols = [c for c in sub.columns if c.startswith("metrics.") and not c.endswith("_total") and sub[c].notna().any()]
    rel = sub.iloc[0][rel_cols].dropna().astype(float)
    rel.index = [c.replace("metrics.", "") for c in rel.index]
    rel_type = pd.Series(["text" if i.startswith("text_") else "numeric" for i in rel.index], index=rel.index)
    top = rel.sort_values(ascending=False).head(20).round(4).to_frame("mean_abs_shap")
    top["type"] = [rel_type[i] for i in top.index]
    top.index = [i.replace("text_", "").replace("num_", "") for i in top.index]
    top.to_csv(f"{RESULT_DIR}/exp3_{ds}.csv")
    # scientific-style .tex: booktabs (hrules) + fette Spaltenheader
    sty = (top.style.format(precision=4, escape="latex")
           .format_index(escape="latex", axis="index").format_index(escape="latex", axis="columns")
           .map_index(lambda v: "font-weight: bold;", axis="columns")
           .hide(axis="index", names=True).hide(axis="columns", names=True))
    sty.to_latex(f"{RESULT_DIR}/exp3_{ds}.tex", hrules=True, convert_css=True, position_float="centering",
                 caption=f"Experiment 3 -- Semantische Relevanz ({ds.replace('_', ' ')}): Top-20 SHAP-Features (Text und numerisch).", label=f"tab:exp3_{ds}")
    print(f"=== Exp 3 – {ds}: Top 20 Features (Text + numerisch) ===")
    display(top)

    # bar chart: top features colored by type
    colors = {"text": "tab:orange", "numeric": "tab:blue"}
    ax = top["mean_abs_shap"][::-1].plot.barh(figsize=(8, 6), color=[colors[t] for t in top["type"][::-1]])
    ax.set_title(f"Exp 3 – {ds}: Top 20 SHAP-Features")
    ax.set_xlabel("mean(|SHAP|)")
    handles = [plt.Rectangle((0, 0), 1, 1, color=colors[t]) for t in ["text", "numeric"]]
    ax.legend(handles, ["text", "numeric"], title="Typ")
    plt.tight_layout()
    plt.savefig(f"{DIAGRAM_DIR}/exp3_shap_{DS_ABBR[ds]}.png", facecolor='white', edgecolor='white', dpi=300)
    plt.savefig(f"{DIAGRAM_DIR}/exp3_shap_{DS_ABBR[ds]}.pdf", facecolor='white', edgecolor='white')
    plt.show()
    plt.close()

    # second chart: total SHAP contribution text vs numeric
    by_type = rel.groupby(rel_type).sum().reindex(["text", "numeric"]).fillna(0).round(4)
    ax = by_type.plot.bar(figsize=(6, 4), rot=0, color=[colors[t] for t in by_type.index])
    ax.set_title(f"Exp 3 – {ds}: SHAP Text vs. numerisch")
    ax.set_ylabel("Summe mean(|SHAP|)")
    ax.set_xlabel("")
    plt.tight_layout()
    plt.savefig(f"{DIAGRAM_DIR}/exp3_shap_sum_{DS_ABBR[ds]}.png", facecolor='white', edgecolor='white', dpi=300)
    plt.savefig(f"{DIAGRAM_DIR}/exp3_shap_sum_{DS_ABBR[ds]}.pdf", facecolor='white', edgecolor='white')
    plt.show()
    plt.close()

## Experiment 4 – TFMs zur binären Klassifikation
- ConTextTab vs. TabPFN-Klassifikation; AUPRC & AUROC je Modell
- Je Modell zwei `distribution`-Varianten (`original`, `balanced_1to4`) als eigene Runs (`*_original` / `*_balanced_1to4`)
- Tabelle je Datensatz; je 1 kombinierter Chart für AUPRC und AUROC (beide Datensätze, Modelle alphabetisch)

In [ ]:
sub = df[df["exp_num"] == 4] if len(df) else df
if not len(sub):
    print("keine Exp-4-Runs")
else:
    # Tabelle je Datensatz
    for ds in datasets:
        s = sub[sub["dataset"] == ds]
        if not len(s):
            continue
        g = s.groupby("model")
        tab = g[["AUPRC", "AUROC", "runtime_s"]].mean()
        tab["n_runs"] = g.size()
        tab = tab.sort_values("AUPRC", ascending=False).round(4)
        save = tab.drop(columns="n_runs")  # n_runs nur zur Anzeige, nicht speichern
        save.to_csv(f"{RESULT_DIR}/exp4_{ds}.csv")
        # scientific-style .tex: booktabs (hrules) + fette Spaltenheader
        sty = (save.style.format(precision=4, escape="latex")
               .format_index(escape="latex", axis="index").format_index(escape="latex", axis="columns")
               .map_index(lambda v: "font-weight: bold;", axis="columns")
               .hide(axis="index", names=True).hide(axis="columns", names=True))
        sty.to_latex(f"{RESULT_DIR}/exp4_{ds}.tex", hrules=True, convert_css=True, position_float="centering",
                     caption=f"Experiment 4 -- TFM-Klassifikation ({ds.replace('_', ' ')}): AUPRC, AUROC und Laufzeit je Modell.", label=f"tab:exp4_{ds}")
        print(f"=== Experiment 4 – {ds} ===")
        display(tab)

    # AUPRC- & AUROC-Chart (beide Datensätze) -> diagrams/
    for metric, label, fname in [("AUPRC", "AUPRC", "auprc"), ("AUROC", "AUROC", "auroc")]:
        piv = sub.pivot_table(index="model", columns="dataset", values=metric, aggfunc="mean").sort_index()
        # Modellname und distribution-Variante auf zwei Zeilen
        piv.index = [m.replace("_balanced_", "\nbalanced_").replace("_original", "\noriginal") for m in piv.index]
        ax = piv.plot.bar(figsize=(8, 4), rot=0)
        ax.set_title(f"Experiment 4 – {label} (beide Datensätze)")
        ax.set_ylabel(label)
        ax.set_xlabel("")
        ax.legend(title="Datensatz")
        plt.tight_layout()
        plt.savefig(f"{DIAGRAM_DIR}/exp4_{fname}.png", facecolor='white', edgecolor='white', dpi=300)
        plt.savefig(f"{DIAGRAM_DIR}/exp4_{fname}.pdf", facecolor='white', edgecolor='white')
        plt.show()
        plt.close()